<a href="https://colab.research.google.com/github/moise97/Extract_-_Structure_Data_from_SDFs_pharmaceutical_documentation/blob/main/Final_Pipeline_Presentation_(personal_Project_chatbot_built).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install system packages: tesseract (OCR engine) and poppler (PDF→image converter)
!apt-get install -q tesseract-ocr poppler-utils

# Core document processing
!pip install -q pdfplumber          # PDF text extraction, preserves layout
!pip install -q pytesseract          # Python wrapper for Tesseract OCR
!pip install -q pdf2image            # Converts PDF pages to PIL images for OCR
!pip install -q Pillow               # Image pre-processing before OCR

# RAG stack
!pip install -q llama-index
!pip install -q llama-index-embeddings-huggingface
!pip install -q llama-index-llms-huggingface
!pip install -q sentence-transformers

# LLM
!pip install -q transformers accelerate sentencepiece

# UI
!pip install -q gradio

print('✅ All packages installed.')

Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 51 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 0s (2,225 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 122412 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4

In [2]:
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
import torch

# TinyLlama: 1.1B params, runs in float16 on T4 GPU (~2.2 GB VRAM)
# Uses ChatML format — reliable for structured JSON prompts
MODEL_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
print(f'Loading {MODEL_NAME}...\n')

llm = HuggingFaceLLM(
    model_name=MODEL_NAME,
    tokenizer_name=MODEL_NAME,
    context_window=2048,       # TinyLlama max context
    max_new_tokens=512,        # cap response length
    generate_kwargs={'temperature': 0.2, 'do_sample': True},  # low temp = factual
    device_map='auto',
    model_kwargs={'torch_dtype': torch.float16}  # no bitsandbytes needed
)

# BGE-small: best retrieval quality at small model size
# Outperformed MiniLM and E5 in earlier module testing
embed_model = HuggingFaceEmbedding(model_name='BAAI/bge-small-en-v1.5')

# Apply globally so all LlamaIndex components use these models
Settings.llm         = llm
Settings.embed_model = embed_model
Settings.chunk_size    = 512   # tokens per chunk
Settings.chunk_overlap = 50    # overlap preserves context at boundaries

print('✅ Models loaded.')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0...



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Models loaded.
   GPU: Tesla T4
   VRAM used: 2.33 GB


In [3]:
import pdfplumber
import pytesseract
from pdf2image import convert_from_path
from PIL import Image, ImageEnhance
import os
from llama_index.core import Document

# --- Doc type keyword map ---
# Used to auto-tag each page with a document category
# This metadata is later used for filtered retrieval
DOC_TYPE_KEYWORDS = {
    'certificate_of_quality':  ['certificate of quality', 'lot number', 'batch', 'sterilization'],
    'bse_tse_declaration':     ['bse', 'tse', 'bovine', 'animal origin'],
    'packaging_specification': ['packaging', 'carton', 'dimensions', 'units per box'],
    'cover_letter':            ['dear', 'enclosed', 'sincerely', 'attached'],
    'material_description':    ['material', 'composition', 'polymer'],
    'packing_list':            ['packing list', 'quantity', 'gross weight'],
    'test_report':             ['test report', 'results', 'pass', 'fail'],
}

def detect_doc_type(text):
    """Keyword-based doc type detection — runs on every page."""
    text_lower = text.lower()
    for dtype, keywords in DOC_TYPE_KEYWORDS.items():
        if any(kw in text_lower for kw in keywords):
            return dtype
    return 'unknown'

def preprocess_for_ocr(img):
    """Enhance image contrast before OCR to improve accuracy on faded scans."""
    enhancer = ImageEnhance.Contrast(img)
    return enhancer.enhance(2.0)

def extract_page_text(page, page_img=None):
    """
    Try digital extraction first (fast, accurate).
    If fewer than 50 chars returned, it's likely a scanned page — fall back to OCR.
    Returns (text, method_used)
    """
    text = (page.extract_text() or '').strip()
    if len(text) < 50 and page_img is not None:
        try:
            enhanced = preprocess_for_ocr(page_img)
            text = pytesseract.image_to_string(enhanced).strip()
            return text, 'ocr'
        except Exception:
            pass
    return text, 'digital'

def process_pdf(filepath):
    """
    Full pipeline for a single PDF:
    1. Convert pages to images (for OCR fallback)
    2. Extract text (digital or OCR)
    3. Detect doc type from content
    4. Build LlamaIndex Documents with rich metadata
    """
    filename = os.path.basename(filepath)
    documents = []
    log = []

    # Convert all pages to images upfront (used only for OCR fallback)
    try:
        page_images = convert_from_path(filepath, dpi=150)
    except Exception:
        page_images = []

    with pdfplumber.open(filepath) as pdf:
        current_type = 'unknown'
        doc_start = 1

        for i, page in enumerate(pdf.pages):
            img = page_images[i] if i < len(page_images) else None
            text, method = extract_page_text(page, img)
            if not text:
                continue

            # Update doc type when a new document header is detected
            detected = detect_doc_type(text)
            if detected != 'unknown':
                current_type = detected
                doc_start = i + 1

            # Attach full metadata to each chunk for filtered retrieval
            documents.append(Document(
                text=text,
                metadata={
                    'source':          filename,
                    'page':            i + 1,
                    'total_pages':     len(pdf.pages),
                    'doc_type':        current_type,
                    'page_in_doc':     (i + 1) - doc_start,
                    'extract_method':  method,
                }
            ))
            log.append(f'  p.{i+1}: {current_type} [{method}] — {text[:60].replace(chr(10)," ")}...')

    return documents, log

print('✅ Document processing pipeline defined.')

✅ Document processing pipeline defined.


In [4]:
from llama_index.core import VectorStoreIndex
from llama_index.core.response_synthesizers import get_response_synthesizer, ResponseMode
import datetime

# Global state: index and file list persist across Gradio interactions
current_index  = None
loaded_files   = []

# System prompt: tells TinyLlama to cite sources and stay grounded
SYSTEM_PROMPT = """You are a precise pharmaceutical document analyst.
Answer questions based ONLY on the provided document context.
Always cite your sources: mention the document filename and page number.
If the answer is not in the context, say: 'This information was not found in the uploaded documents.'
Be concise and factual. Do not speculate beyond the provided text."""

def query_rag(question, top_k=4):
    """
    Full RAG pipeline:
      1. Retrieve top-k most similar chunks from the vector index
      2. Calculate confidence = average similarity score across retrieved nodes
      3. Build source list with file, page, doc_type, and text excerpt
      4. Synthesize answer using TinyLlama with source-citing system prompt
    Returns a dict with answer, sources, confidence, chunk_count
    """
    if current_index is None:
        return {'answer':'⚠️ No documents loaded.','sources':[],'confidence':0.0,'chunk_count':0}

    # Step 1: Retrieve
    retriever = current_index.as_retriever(similarity_top_k=top_k)
    nodes = retriever.retrieve(question)

    if not nodes:
        return {'answer':'No relevant content found.','sources':[],'confidence':0.0,'chunk_count':0}

    # Step 2: Confidence score — average similarity of retrieved chunks
    scores = [n.score for n in nodes if n.score]
    confidence = round(sum(scores)/len(scores), 3) if scores else 0.0

    # Step 3: Deduplicated source list
    sources, seen = [], set()
    for node in nodes:
        m = node.metadata
        key = f"{m.get('source')}|{m.get('page')}"
        if key not in seen:
            seen.add(key)
            sources.append({
                'file':    m.get('source','?'),
                'page':    m.get('page','?'),
                'type':    m.get('doc_type','?'),
                'method':  m.get('extract_method','?'),
                'score':   round(node.score,3) if node.score else 0,
                'excerpt': node.get_text()[:120].replace('\n',' ')
            })

    # Step 4: Synthesize — inject system prompt into the question
    synthesizer = get_response_synthesizer(response_mode=ResponseMode.COMPACT)
    augmented   = f"{SYSTEM_PROMPT}\n\nUser question: {question}"
    response    = synthesizer.synthesize(augmented, nodes=nodes)

    return {
        'answer':      response.response.strip(),
        'sources':     sources,
        'confidence':  confidence,
        'chunk_count': len(nodes)
    }

print('✅ RAG engine defined.')

✅ RAG engine defined.


In [5]:
import gradio as gr
import tempfile

# --- UI formatting helpers ---

def confidence_bar(score):
    """Render confidence as a visual ASCII bar + percentage label."""
    pct   = int(score * 100)
    filled = int(pct / 10)
    bar   = '█' * filled + '░' * (10 - filled)
    label = 'High' if pct >= 70 else 'Medium' if pct >= 40 else 'Low'
    return f'Confidence: [{bar}] {pct}% ({label})'

def format_sources(sources):
    """Format source citations for the sources panel."""
    if not sources:
        return 'No sources found.'
    lines = []
    for i, s in enumerate(sources, 1):
        lines.append(
            f"[{i}] {s['file']} — Page {s['page']} | "
            f"Type: {s['type']} | Score: {s['score']} | Extract: {s['method']}\n"
            f"    ↳ \"{s['excerpt']}...\""
        )
    return '\n'.join(lines)

# --- Event handlers ---

def handle_upload(pdf_files):
    """Process uploaded PDFs: extract, tag, and build the vector index."""
    global current_index, loaded_files
    if not pdf_files:
        yield '⚠️ No files selected.', ''
        return

    yield '⏳ Extracting text and running OCR where needed...', ''
    all_docs, loaded_files, log_lines = [], [], []

    for f in pdf_files:
        fname = os.path.basename(f.name)
        loaded_files.append(fname)
        docs, page_log = process_pdf(f.name)
        all_docs.extend(docs)
        log_lines.append(f'📄 {fname} — {len(docs)} pages')
        log_lines.extend(page_log)

    yield '⏳ Building vector index...', '\n'.join(log_lines)
    current_index = VectorStoreIndex.from_documents(all_docs)
    yield (
        f'✅ Ready — {len(all_docs)} pages indexed from {len(loaded_files)} file(s): {", ".join(loaded_files)}',
        '\n'.join(log_lines)
    )

def handle_chat(user_msg, history, top_k):
    """Query the RAG pipeline and update the chat and sources panels."""
    if not user_msg.strip():
        yield history, '', 'Ask a question to see sources.', ''
        return

    # Immediately show thinking indicator — uses generator pattern (yield)
    history = history + [[user_msg, '⏳ Searching documents...']]
    yield history, '', 'Retrieving...', ''

    result     = query_rag(user_msg, top_k=int(top_k))
    conf_line  = confidence_bar(result['confidence'])
    chunk_line = f"Chunks used: {result['chunk_count']}"

    # Replace thinking message with real answer + metadata footer
    history[-1][1] = f"{result['answer']}\n\n{conf_line} | {chunk_line}"

    stats = (
        f"Confidence : {result['confidence']}\n"
        f"Chunks used: {result['chunk_count']}\n"
        f"Top-k      : {top_k}"
    )
    yield history, '', format_sources(result['sources']), stats

def save_history(history):
    """Export full chat conversation to a downloadable .txt file."""
    if not history:
        return None
    lines = [
        f'DocChat Pro — Export {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}',
        f'Files: {", ".join(loaded_files)}', '='*60, ''
    ]
    for i, (q, a) in enumerate(history, 1):
        lines += [f'Q{i}: {q}', f'A{i}: {a}', '']
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix='.txt', mode='w')
    tmp.write('\n'.join(lines))
    tmp.close()
    return tmp.name

# --- Build the Gradio UI ---

with gr.Blocks(
    title='DocChat Pro',
    theme=gr.themes.Soft(primary_hue='violet', neutral_hue='slate'),
    css='footer{display:none!important} .chatbot{font-size:14px}'
) as demo:

    gr.Markdown('# 🤖 DocChat Pro\n**Ask questions about your PDFs — powered by TinyLlama + RAG**')

    with gr.Row():

        # Left panel: upload + settings + export
        with gr.Column(scale=1, min_width=280):
            gr.Markdown('### 📁 Documents')
            pdf_input     = gr.File(label='Upload PDFs', file_types=['.pdf'], file_count='multiple')
            upload_btn    = gr.Button('📥 Process & Index', variant='primary')
            upload_status = gr.Textbox(label='Status', value='No documents loaded.', interactive=False, lines=2)
            upload_log    = gr.Textbox(label='Processing log', interactive=False, lines=7)

            gr.Markdown('### ⚙️ Settings')
            top_k_slider = gr.Slider(minimum=1, maximum=8, value=4, step=1,
                label='Chunks to retrieve (top-k)',
                info='Higher = more context, slower response')

            gr.Markdown('### 💾 Export')
            save_btn      = gr.Button('⬇️ Download chat history')
            download_file = gr.File(label='', visible=False)

        # Right panel: chat + sources + stats
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(label='Chat', height=400, bubble_full_width=False, show_copy_button=True)

            with gr.Row():
                msg_box  = gr.Textbox(placeholder='Ask a question about your documents...', label='', scale=5)
                send_btn = gr.Button('Send', variant='primary', scale=1)

            clear_btn = gr.Button('🗑️ Clear', size='sm')

            with gr.Row():
                sources_box = gr.Textbox(label='📄 Retrieved sources', interactive=False, lines=5)
                stats_box   = gr.Textbox(label='📊 Query stats',       interactive=False, lines=4)

            gr.Examples(
                examples=[
                    ['What is this document about?'],
                    ['What sterilization method was used?'],
                    ['What are the storage conditions?'],
                    ['List all part numbers or lot numbers.'],
                    ['Is there a BSE or TSE declaration?'],
                    ['Summarize the key specifications.'],
                ],
                inputs=msg_box, label='Example questions'
            )

    # Wire all events
    upload_btn.click(fn=handle_upload, inputs=pdf_input, outputs=[upload_status, upload_log])
    send_btn.click(fn=handle_chat, inputs=[msg_box, chatbot, top_k_slider], outputs=[chatbot, msg_box, sources_box, stats_box])
    msg_box.submit(fn=handle_chat, inputs=[msg_box, chatbot, top_k_slider], outputs=[chatbot, msg_box, sources_box, stats_box])
    clear_btn.click(fn=lambda: ([], '', 'Ask a question.', ''), outputs=[chatbot, msg_box, sources_box, stats_box])
    save_btn.click(fn=lambda h: gr.File(value=save_history(h), visible=True) if h else gr.File(visible=False), inputs=chatbot, outputs=download_file)

# share=True generates a public link valid for 72 hours — open on your phone to test
demo.launch(share=True, debug=False)

/tmp/ipykernel_2454/86806048.py:95: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_2454/86806048.py:95: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_2454/86806048.py:124: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label='Chat', height=400, bubble_full_width=False, show_copy_button=True)
/tmp/ipykernel_2454/86806048.py:124: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use '

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cc17e7e0897c602f7e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
